# 의회 회의록 JSON → Parquet 변환 안내

2024년과 2025년의 JSON 회의록을 같은 규칙으로 변환하는 노트북입니다. 원본의 HTML은 입력 JSON에 보관하고, 출력에는 정리한 본문 `MINTS_TEXT`, 회의 날짜, 원본 파일명과 행 번호를 함께 저장합니다.

**실행 순서:** 아래의 다섯 코드 셀을 **1 → 2 → 3 → 4 → 5** 순서로 실행합니다. 1단계에서 경로와 설정을 확인하고, 실제 파일 변환은 마지막 5단계에서 시작합니다.

이번 설명본은 기존 코드에 함수별 `#` 주석과 기능 안내만 추가했습니다. 기존 설정·함수 본문·코드 셀 순서·메타정보·출력·실행 번호는 유지했으며, 주석을 작성하는 과정에서 실제 데이터 변환을 실행하지 않았습니다. 남아 있는 셀 출력과 실행 번호는 기존 노트북의 기록입니다.


## 1. 실행 환경과 입출력 경로 설정

**입력 →** 2024년·2025년 원본 폴더 경로, 출력 폴더 경로, 기존 결과 덮어쓰기 여부, DuckDB 메모리 제한입니다.

**처리 →** 필요한 모듈을 불러오고 `INPUT_DIRS`에 두 연도의 입력 폴더를 묶습니다. `OVERWRITE = False`이면 나중에 변환할 때 이미 있는 Parquet 파일을 건너뜁니다.

**결과 →** 뒤의 셀들이 참조할 경로와 공통 설정이 준비됩니다. 이 셀에서는 JSON을 읽거나 Parquet을 생성하지 않습니다.

**실행 순서:** 첫 번째로 실행하세요. 현재 코드의 경로값을 확인한 다음 2단계로 진행합니다.


In [1]:
#!/usr/bin/env python3
"""지정된 2024년·2025년 의회 회의록 폴더를 DuckDB로 변환합니다.

본문은 MINTS_TEXT만 저장합니다. MINTS_HTML 원본은 입력 JSON에 보관됩니다.

설치: python3 -m pip install duckdb numpy
실행: 노트북의 모든 셀을 위에서 아래로 실행하세요.
"""

import os
import re
from html.parser import HTMLParser
from pathlib import Path
from uuid import uuid4

import duckdb

# 사용자께서 지정한 실제 경로입니다.
INPUT_DIR_2024 = Path("/Users/kimhyunjin/Desktop/kimhyunjin project/2026 데이터 AI 분석 경진대회/data/raw_data/2024_광역 회의록")
INPUT_DIR_2025 = Path("/Users/kimhyunjin/Desktop/kimhyunjin project/2026 데이터 AI 분석 경진대회/data/raw_data/2025_광역 회의록")
OUTPUT_DIR = Path("/Users/kimhyunjin/Desktop/kimhyunjin project/2026 데이터 AI 분석 경진대회/data/format_changed_data")

# 기존 입력·출력 경로를 유지하며 두 연도를 순서대로 처리합니다.
INPUT_DIRS = {
    2024: INPUT_DIR_2024,
    2025: INPUT_DIR_2025,
}

# 기존 결과를 다시 만들려면 True로 변경하세요.
OVERWRITE = False
MEMORY_LIMIT = "16GB"



## 2. HTML 본문 추출 함수 정의

**입력 →** 회의록의 `MINTS_HTML` 문자열 또는 `None`입니다.

**처리 →** `MinutesTextParser`가 문단·줄바꿈 경계를 기록하고, `script`·`style`·`head` 영역의 텍스트를 제외합니다. 인라인 태그 자체로는 문장을 끊지 않으며, 마지막에 줄 내부 공백과 빈 줄을 정리합니다. `html_to_text`가 이 과정을 문자열 하나에 적용합니다.

**결과 →** SQL에서 호출할 본문 변환 함수가 정의됩니다. 실제 호출 시 정리된 문자열을 반환하고, 결측값 `None`은 그대로 유지합니다. 내용 요약이나 키워드 추출을 수행하는 단계는 아닙니다.

**실행 순서:** 1단계 다음에 실행하세요. 여기서는 클래스와 함수를 정의하며 실제 JSON 파일은 읽지 않습니다.


In [2]:
class MinutesTextParser(HTMLParser):
    """문단 경계는 보존하고, 인라인 태그는 문장을 끊지 않고 처리합니다."""

    BLOCK_TAGS = {
        "address", "article", "aside", "blockquote", "dd", "div", "dl", "dt",
        "fieldset", "figcaption", "figure", "footer", "form", "h1", "h2", "h3",
        "h4", "h5", "h6", "header", "hr", "li", "main", "nav", "ol", "p",
        "pre", "section", "table", "tr", "ul",
    }
    SKIP_TAGS = {"script", "style", "head"}

    # 기능: HTML 본문을 모을 버퍼와 제외 영역의 중첩 깊이를 초기화합니다.
    # 입력: 새로 생성하는 파서 인스턴스(self)이며 별도의 사용자 인자는 없습니다.
    # 반환/부작용: None을 반환하며 문자 참조 자동 변환을 켜고 parts와 skip_depth 상태를 만듭니다.
    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self.parts: list[str] = []
        self.skip_depth = 0

    # 기능: 시작 태그에 따라 제외 영역을 표시하고 문단·줄바꿈·표 셀의 경계를 기록합니다.
    # 입력: HTMLParser가 전달하는 태그명(tag)과 속성 목록(attrs)이며 속성값은 현재 사용하지 않습니다.
    # 반환/부작용: None을 반환하며 skip_depth를 갱신하거나 parts에 줄바꿈·탭을 추가합니다.
    def handle_starttag(self, tag: str, attrs: list) -> None:
        if tag in self.SKIP_TAGS:
            self.skip_depth += 1
        if self.skip_depth:
            return
        if tag == "br" or tag in self.BLOCK_TAGS:
            self.parts.append("\n")
        elif tag in {"td", "th"}:
            self.parts.append("\t")

    # 기능: 제외 영역의 끝을 처리하고 본문의 블록 태그가 끝나면 문단 경계를 기록합니다.
    # 입력: HTMLParser가 전달하는 종료 태그명(tag)입니다.
    # 반환/부작용: None을 반환하며 skip_depth를 줄이거나 parts에 줄바꿈을 추가합니다.
    def handle_endtag(self, tag: str) -> None:
        if tag in self.SKIP_TAGS:
            self.skip_depth = max(0, self.skip_depth - 1)
            return
        if not self.skip_depth and tag in self.BLOCK_TAGS:
            self.parts.append("\n")

    # 기능: 닫힘 표시가 포함된 단독 태그를 기존 시작·종료 태그 처리 규칙으로 처리합니다.
    # 입력: 단독 태그명(tag)과 속성 목록(attrs)입니다.
    # 반환/부작용: None을 반환하며 두 처리 메서드를 호출해 parts와 skip_depth를 갱신할 수 있습니다.
    def handle_startendtag(self, tag: str, attrs: list) -> None:
        self.handle_starttag(tag, attrs)
        self.handle_endtag(tag)

    # 기능: 제외 대상인 script·style·head 영역 밖의 텍스트 조각만 본문 버퍼에 모읍니다.
    # 입력: HTMLParser가 전달하는 텍스트 조각(data)입니다.
    # 반환/부작용: None을 반환하며 제외 영역 밖일 때만 parts에 텍스트를 추가합니다.
    def handle_data(self, data: str) -> None:
        if not self.skip_depth:
            self.parts.append(data)

    # 기능: 모아 둔 본문을 합치고 줄 내부의 연속 공백과 빈 줄을 정리합니다.
    # 입력: 현재 파서 인스턴스에 저장된 parts를 사용하며 별도의 인자는 없습니다.
    # 반환/부작용: 줄바꿈으로 연결한 정제 문자열을 반환하며 파일을 쓰거나 parts를 변경하지 않습니다.
    def text(self) -> str:
        raw = "".join(self.parts).replace("\xa0", " ")
        lines = (re.sub(r"[^\S\r\n]+", " ", line).strip() for line in raw.splitlines())
        return "\n".join(line for line in lines if line)


# 기능: HTML 문자열 하나를 전용 파서로 처리하여 태그 없는 회의록 본문으로 바꿉니다.
# 입력: HTML 문자열(value) 또는 결측값 None입니다.
# 반환/부작용: 정제 문자열을 반환하고 입력이 None이면 None을 유지하며 파일은 변경하지 않습니다.
def html_to_text(value: str | None) -> str | None:
    if value is None:
        return None
    parser = MinutesTextParser()
    parser.feed(value)
    parser.close()
    return parser.text()

## 3. JSON → Parquet 변환 쿼리 정의

**입력 →** 실행 시 전달받을 JSON 경로 `$1`, 임시 Parquet 경로 `$2`, 원본 JSON 파일명 `$3`입니다.

**처리 →** DuckDB가 JSON 구조를 읽어 기존 열 중 `MINTS_HTML`을 제외합니다. `MTG_DE`를 날짜형 `MTG_DATE`로 변환하고, HTML에서 `MINTS_TEXT`를 추출하며, 추적용 `SOURCE_FILE`과 1부터 시작하는 `SOURCE_ROW`를 추가합니다. 날짜 해석에 실패하면 `MTG_DATE`는 결측값이 됩니다.

**결과 →** ZSTD 압축 Parquet으로 저장하는 `QUERY` 문자열이 정의됩니다. 이 셀은 쿼리를 작성해 두는 단계이며 파일 변환은 아직 실행하지 않습니다.

**실행 순서:** 2단계 다음에 실행하세요. 4단계의 변환 함수가 이 쿼리를 사용합니다.

**입력 열 조건:** 현재 쿼리는 `MINTS_HTML`과 `MTG_DE` 열이 있는 JSON을 전제로 합니다. 빈 배열은 뒤 단계에서 건너뛰지만, 비어 있지 않은 파일에 해당 열이 없으면 변환 오류가 발생합니다.


In [3]:
QUERY = """
    COPY (
        SELECT
            * EXCLUDE (MINTS_HTML),
            CAST(try_strptime(CAST(MTG_DE AS VARCHAR), '%Y%m%d') AS DATE) AS MTG_DATE,
            html_to_text(CAST(MINTS_HTML AS VARCHAR)) AS MINTS_TEXT,
            $3::VARCHAR AS SOURCE_FILE,
            row_number() OVER () AS SOURCE_ROW
        FROM read_json_auto(
            $1,
            format = 'auto',
            sample_size = -1,
            maximum_object_size = 134217728,
            ignore_errors = false
        )
    ) TO $2 (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    );
"""

## 4. 빈 배열 검사와 폴더별 변환 함수 정의

**입력 →** 하나의 입력 폴더 `Path`와 출력 폴더 `Path`입니다. `is_empty_json_array`는 개별 파일이 빈 배열 `[]`인지 확인합니다. 길이가 0인 파일은 빈 배열로 판정하지 않습니다.

**처리 →** `main`은 입력 폴더 바로 아래의 `*.json`을 파일명 순서로 처리합니다. 기존 결과는 덮어쓰기 설정에 따라, 빈 배열은 항상 건너뜁니다. 나머지는 임시 Parquet으로 변환한 뒤 저장 행 수를 확인합니다. 확인에 성공하면 최종 파일명으로 옮기며, 파일별 오류는 기록하고 다음 파일을 계속 처리합니다.

**결과 →** 실행 시 변환된 Parquet 파일과 진행 로그가 생성됩니다. 파일별 실패가 없으면 `0`, 있으면 `1`을 반환하며, 입력 폴더 오류나 초기화 오류는 호출한 쪽으로 전달될 수 있습니다.

**실행 순서:** 3단계 다음에 실행하세요. 이 셀은 함수를 정의하는 단계이며 실제 변환은 5단계에서 호출합니다.

**현재 구현의 주의점:** `main`의 `input_dir` 기본값은 `Path`가 아닌 `INPUT_DIRS` 사전입니다. 따라서 `main()`만 호출하면 폴더 검사에서 오류가 발생합니다. 마지막 셀은 각 연도의 `Path`를 `input_dir=`에 명시하여 호출합니다. 설명 추가 범위에 맞춰 이 기본값은 수정하지 않았습니다.


In [5]:
# 기능: 파일 내용이 공백과 선택적인 UTF-8 BOM을 제외하고 빈 JSON 배열 []뿐인지 확인합니다.
# 입력: 확인할 JSON 파일의 경로(path)입니다.
# 반환/부작용: 빈 배열이면 True, 그 외에는 False를 반환하며 파일을 8,192자씩 읽기만 합니다.
def is_empty_json_array(path: Path) -> bool:
    """내용이 빈 JSON 배열([])인 경우만 True. 비어 있지 않은 파일은 초반에 반환합니다."""
    position = 0
    with path.open("r", encoding="utf-8-sig") as stream:
        while chunk := stream.read(8192):
            for char in chunk:
                if char in " \t\r\n":
                    continue
                if position >= 2 or char != "[]"[position]:
                    return False
                position += 1
    return position == 2


# 기능: 단일 입력 폴더의 JSON을 순서대로 Parquet으로 변환하고 저장 행 수를 확인합니다.
# 입력: 입력 폴더 Path(input_dir)와 출력 폴더 Path(output_dir)이며 공통 변환 설정도 참조합니다.
# 반환: 파일별 변환 실패가 없으면 0, 있으면 1을 반환하며 폴더·초기화 오류는 예외로 전달될 수 있습니다.
# 부작용: 출력 폴더·임시 파일·Parquet을 만들고 진행 로그를 출력하며 임시 파일과 DB 연결을 정리합니다.
def main(input_dir: Path = INPUT_DIRS, output_dir: Path = OUTPUT_DIR) -> int:
    if not input_dir.is_dir():
        raise FileNotFoundError(f"입력 폴더를 찾을 수 없습니다: {input_dir}")

    # 현재 폴더의 JSON만 처리합니다. 하위 폴더는 탐색하지 않습니다.
    json_files = sorted(input_dir.glob("*.json"))
    print(f"발견된 JSON 파일 수: {len(json_files)}개")
    if not json_files:
        raise FileNotFoundError(f"JSON 파일이 없습니다: {input_dir}")
    output_dir.mkdir(parents=True, exist_ok=True)

    con = duckdb.connect()
    success = failures = skipped = empty_files = total_rows = 0
    try:
        con.execute(f"SET memory_limit = '{MEMORY_LIMIT}';")
        con.execute("SET preserve_insertion_order = true;")
        con.create_function(
            "html_to_text", html_to_text, ["VARCHAR"], "VARCHAR", null_handling="special"
        )
        for json_file in json_files:
            parquet_file = output_dir / f"{json_file.stem}.parquet"
            if parquet_file.exists() and not OVERWRITE:
                print(f"[건너뜀] 기존 파일: {parquet_file.name}")
                skipped += 1
                continue

            # 성공적으로 저장·검증된 파일만 최종 파일명으로 옮깁니다.
            temporary = output_dir / f".{parquet_file.name}.{uuid4().hex}.tmp"
            try:
                if is_empty_json_array(json_file):
                    print(f"[건너뜀] 빈 JSON 배열: {json_file.name} (회의록 0건)")
                    empty_files += 1
                    continue
                print(f"[변환 중] {json_file.name} -> {parquet_file.name}")
                copied_rows = con.execute(
                    QUERY, [str(json_file), str(temporary), json_file.name]
                ).fetchone()[0]
                saved_rows, invalid_dates = con.execute(
                    "SELECT count(*), count(*) FILTER (WHERE MTG_DATE IS NULL) "
                    "FROM read_parquet($1)",
                    [str(temporary)],
                ).fetchone()
                if saved_rows != copied_rows:
                    raise RuntimeError(f"행 수 불일치: 변환 {copied_rows}, 저장 {saved_rows}")
                os.replace(temporary, parquet_file)
                success += 1
                total_rows += saved_rows
                print(f"[완료] {parquet_file.name}: {saved_rows:,}행")
                if invalid_dates:
                    print(f"  [주의] 날짜 변환 불가 {invalid_dates:,}건: 원본 MTG_DE를 확인하세요.")
            except Exception as error:
                failures += 1
                print(f"[실패] {json_file.name}: {error}")
            finally:
                temporary.unlink(missing_ok=True)
    finally:
        con.close()

    print(
        f"변환 결과: 성공 {success}개 / 실패 {failures}개 / "
        f"기존 파일 건너뜀 {skipped}개 / 빈 JSON 건너뜀 {empty_files}개 / "
        f"저장 {total_rows:,}행"
    )
    return 1 if failures else 0


## 5. 두 연도 변환 실행과 결과 확인

**입력 →** 앞서 정의한 `INPUT_DIRS`, `OUTPUT_DIR`, 변환 함수와 공통 설정입니다.

**처리 →** 2024년과 2025년의 입력 경로를 각각 `main(input_dir=..., output_dir=...)`에 전달합니다. 한 연도의 호출에서 예외가 발생해도 실패 코드를 기록하고 다음 연도를 계속 처리합니다.

**결과 →** 실제 Parquet 저장이 실행되며, 연도별 종료 코드가 `conversion_results`에 남습니다. 두 연도 모두 코드 `0`이면 `exit_code`는 `0`, 하나라도 실패하면 `1`입니다. 기존 파일·빈 배열을 건너뛴 경우도 정상 종료에 포함되므로 실제 변환 건수는 위의 로그를 함께 확인하세요.

**실행 순서:** 1~4단계를 순서대로 실행한 뒤 마지막으로 실행하세요. 이 셀부터 설정한 데이터 폴더를 읽고 출력 폴더에 결과를 저장합니다.


In [6]:
# 공통 설정·함수 셀을 먼저 실행하세요.
# 2024년과 2025년을 같은 변환 규칙으로 처리합니다.
conversion_results = {}
for year, input_dir in INPUT_DIRS.items():
    print(f"\n[{year}년 변환 시작]")
    print(f"입력 폴더: {input_dir}")
    try:
        conversion_results[year] = main(input_dir=input_dir, output_dir=OUTPUT_DIR)
    except Exception as error:
        # 폴더 오류 등으로 한 연도가 실패해도 다음 연도는 계속 처리합니다.
        conversion_results[year] = 1
        print(f"[{year}년 실패] {error}")

# 노트북에서는 SystemExit를 발생시키지 않고 결과를 변수로 보관합니다.
exit_code = int(any(code != 0 for code in conversion_results.values()))
print("\n연도별 실행 결과")
for year, code in conversion_results.items():
    status = "정상 종료(기존 파일·빈 JSON 건너뛰기 포함)" if code == 0 else "실패 있음 — 위 로그 확인"
    print(f"{year}년: {status}")
print(f"전체 종료 코드: {exit_code} (0=정상, 1=실패 있음)")



[2024년 변환 시작]
입력 폴더: /Users/kimhyunjin/Desktop/김현진 project/2026 데이터 AI 분석 경진대회/data/raw_data/2024_광역 회의록
발견된 JSON 파일 수: 18개
[변환 중] 2024_강원특별자치도의회.json -> 2024_강원특별자치도의회.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[완료] 2024_강원특별자치도의회.parquet: 280행
[변환 중] 2024_경기도의회.json -> 2024_경기도의회.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[완료] 2024_경기도의회.parquet: 378행
[변환 중] 2024_경상남도의회.json -> 2024_경상남도의회.parquet


...

[완료] 2025_제주특별자치도의회.parquet: 398행
[변환 중] 2025_충청남도의회.json -> 2025_충청남도의회.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[완료] 2025_충청남도의회.parquet: 423행
[변환 중] 2025_충청북도의회.json -> 2025_충청북도의회.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[완료] 2025_충청북도의회.parquet: 160행
변환 결과: 성공 17개 / 실패 0개 / 기존 파일 건너뜀 0개 / 빈 JSON 건너뜀 1개 / 저장 6,007행

연도별 실행 결과
2024년: 정상 종료(기존 파일·빈 JSON 건너뛰기 포함)
2025년: 정상 종료(기존 파일·빈 JSON 건너뛰기 포함)
전체 종료 코드: 0 (0=정상, 1=실패 있음)
